In [1]:
"""
=============================================================================
  SPC-DT  —  Shifted Paired Coordinates – Decision Tree Visualization
=============================================================================

WHAT IS SPC-DT?
───────────────
Shifted Paired Coordinates (SPC) is an n-D visualization technique by
Kovalerchuk et al. that maps an n-dimensional point to a sequence of
2D coordinate panels by pairing consecutive features:

    (x0,x1) → Panel 1    (x2,x3) → Panel 2   ...

Every sample appears as a DOT in each panel, and adjacent panel-dots are
connected by a LINE, making a polyline that encodes the full n-D profile.
The "shifted" means the panels are placed side-by-side along the canvas.

SPC-DT adds the Decision Tree on top:
    • Each leaf node in the tree is an axis-aligned hyper-rectangle.
    • Projected onto each 2D panel, that becomes an axis-aligned rectangle.
    • Rectangles are coloured by predicted class; opacity ∝ confidence.

For your 8-feature input [xA | xB]:
    Panel 1: (A_alco,  A_dep)   Panel 2: (A_life,  A_crim)
    Panel 3: (B_alco,  B_dep)   Panel 4: (B_life,  B_crim)

HOW TO READ IT:
───────────────
  • Blue region  = tree predicts Patient A wins in that zone
  • Orange region= tree predicts Patient B wins
  • Darker/more opaque = leaf is more confident
  • Blue dots    = true label is A-wins
  • Orange dots  = true label is B-wins
  • Connecting lines = same sample across all 4 panels
  • Overlapping blue dot inside orange region = misclassification

Reference:
  Kovalerchuk, B., & Delizy, F. (2005). Visual data mining using
  shifted paired coordinates. Proc. SPIE 5764, Visualization and
  Data Analysis, 154–165.

=============================================================================
"""

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from sklearn.tree import DecisionTreeClassifier, _tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
FEATURE_NAMES = ['A_alco', 'A_dep', 'A_life', 'A_crim',
                  'B_alco', 'B_dep', 'B_life', 'B_crim']

# SPC panel definitions: (x_feature_index, y_feature_index)
SPC_PANELS = [(0, 1), (2, 3), (4, 5), (6, 7)]

PANEL_TITLES = [
    'Panel 1\nA: alco vs dep',
    'Panel 2\nA: life vs crim',
    'Panel 3\nB: alco vs dep',
    'Panel 4\nB: life vs crim',
]

# Class colours: 0=A wins (blue), 1=B wins (orange)
CLS_COLOR = {0: '#1976D2', 1: '#E65100'}
CLS_LABEL = {0: 'A wins (y=0)', 1: 'B wins (y=1)'}


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: EXTRACT LEAF BOUNDING BOXES
# ─────────────────────────────────────────────────────────────────────────────

def get_leaf_boxes(clf, X_ref):
    """
    Walk the fitted decision tree depth-first and collect, for every leaf,
    the axis-aligned bounding box (lo[f], hi[f]) for each feature f.

    The bounding box is the intersection of all split constraints on the
    path from the root to that leaf, initialised to the data's feature range.

    Returns
    -------
    list of dicts, each containing:
        lo         : np.ndarray (n_features,) — lower bounds
        hi         : np.ndarray (n_features,) — upper bounds
        predicted  : int   — majority class
        confidence : float — fraction of leaf samples in majority class
        n_samples  : int   — training samples at this leaf
    """
    t       = clf.tree_
    n_feat  = X_ref.shape[1]
    margin  = (X_ref.max(0) - X_ref.min(0)) * 0.05

    # Initialise bounds to the full data range (plus a small margin)
    global_lo = X_ref.min(0) - margin
    global_hi = X_ref.max(0) + margin

    boxes = []

    def walk(node, lo, hi):
        feat = t.feature[node]

        # Leaf node — record the bounding box
        if feat == _tree.TREE_UNDEFINED:
            val  = t.value[node][0]               # class counts
            pred = int(np.argmax(val))
            conf = float(val[pred] / val.sum())
            boxes.append({
                'lo'        : lo.copy(),
                'hi'        : hi.copy(),
                'predicted' : pred,
                'confidence': conf,
                'n_samples' : int(t.n_node_samples[node]),
            })
            return

        thresh = float(t.threshold[node])
        feat   = int(feat)

        # Left branch  (feature <= threshold): tighten upper bound
        hi_l       = hi.copy()
        hi_l[feat] = min(hi_l[feat], thresh)
        walk(t.children_left[node], lo.copy(), hi_l)

        # Right branch (feature >  threshold): tighten lower bound
        lo_r       = lo.copy()
        lo_r[feat] = max(lo_r[feat], thresh)
        walk(t.children_right[node], lo_r, hi.copy())

    walk(0, global_lo.copy(), global_hi.copy())
    return boxes


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: FEATURE NORMALISATION  (maps data to [0, 1] per feature)
# ─────────────────────────────────────────────────────────────────────────────

def fit_normaliser(X):
    mn  = X.min(0)
    rng = X.max(0) - X.min(0)
    rng[rng == 0] = 1.0          # guard against zero-range features
    return mn, rng

def normalise(X, mn, rng):
    return (X - mn) / rng

def normalise_box(lo, hi, mn, rng):
    return (lo - mn) / rng, (hi - mn) / rng


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: COORDINATE CONVERSION HELPER
# ─────────────────────────────────────────────────────────────────────────────

def data_to_fig(ax, xdata, ydata):
    """
    Convert a data-space (xdata, ydata) to figure-fraction coordinates.
    Uses the manually specified axes position + known axis limits.
    Does NOT require fig.canvas.draw().
    """
    pos = ax.get_position()          # Bbox of this axes in figure coords
    xl  = ax.get_xlim()
    yl  = ax.get_ylim()
    xf  = pos.x0 + pos.width  * (xdata - xl[0]) / (xl[1] - xl[0])
    yf  = pos.y0 + pos.height * (ydata - yl[0]) / (yl[1] - yl[0])
    return xf, yf


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: MAIN SPC-DT PLOT
# ─────────────────────────────────────────────────────────────────────────────

def plot_spc_dt(clf, X, y,
                X_train=None, y_train=None,
                X_test=None,  y_test=None,
                save_path='spc_dt_visualization.png'):
    """
    Produce the full SPC-DT visualisation.

    Parameters
    ----------
    clf          : fitted DecisionTreeClassifier
    X            : full feature matrix  (N, 8)
    y            : full labels           (N,)
    X_train/test : optional — used for accuracy annotation only
    save_path    : output PNG file path
    """
    # Normalise all data to [0,1] using the full dataset's range
    mn, rng   = fit_normaliser(X)
    X_norm    = normalise(X, mn, rng)
    leaf_boxes = get_leaf_boxes(clf, X)

    n_panels = len(SPC_PANELS)

    # ── Figure & panel layout ─────────────────────────────────────────────
    # Panels are placed manually so we can draw cross-panel connector lines
    # at the figure level.
    fig = plt.figure(figsize=(22, 6.5))
    fig.patch.set_facecolor('#f7f7f7')

    P_W  = 0.185    # panel width  (figure fraction)
    P_H  = 0.70     # panel height
    P_B  = 0.15     # bottom margin
    P_G  = 0.024    # gap between panels
    P_L0 = 0.040    # left edge of first panel

    axes = []
    for i in range(n_panels):
        left = P_L0 + i * (P_W + P_G)
        ax   = fig.add_axes([left, P_B, P_W, P_H])
        axes.append(ax)

    # ── Draw each panel ───────────────────────────────────────────────────
    for pidx, (xi, yi) in enumerate(SPC_PANELS):
        ax = axes[pidx]
        ax.set_facecolor('#ffffff')
        ax.set_xlim(-0.06, 1.06)
        ax.set_ylim(-0.06, 1.06)
        ax.set_xlabel(FEATURE_NAMES[xi], fontsize=9.5, labelpad=4, color='#444')
        ax.set_ylabel(FEATURE_NAMES[yi], fontsize=9.5, labelpad=4, color='#444')
        ax.set_title(PANEL_TITLES[pidx], fontsize=9, pad=6,
                     fontweight='bold', color='#333')
        ax.tick_params(labelsize=7.5, colors='#666')
        ax.grid(True, alpha=0.20, linewidth=0.4, color='#999')
        for sp in ax.spines.values():
            sp.set_linewidth(0.5)
            sp.set_color('#cccccc')

        # ── 1. Decision region rectangles (from leaf boxes) ───────────────
        for box in leaf_boxes:
            lo_n, hi_n = normalise_box(box['lo'], box['hi'], mn, rng)
            x0, x1 = lo_n[xi], hi_n[xi]
            y0, y1 = lo_n[yi], hi_n[yi]
            w, h   = x1 - x0, y1 - y0
            if w <= 0 or h <= 0:
                continue                          # degenerate box; skip

            c   = CLS_COLOR[box['predicted']]
            # Opacity: base 0.08, scaled up to 0.32 with confidence
            alp = 0.08 + 0.24 * box['confidence']

            # Filled region
            ax.add_patch(mpatches.Rectangle(
                (x0, y0), w, h,
                facecolor=c, edgecolor='none', alpha=alp, zorder=1
            ))
            # Border for high-confidence leaves (≥ 80%)
            if box['confidence'] >= 0.80:
                ax.add_patch(mpatches.Rectangle(
                    (x0, y0), w, h,
                    facecolor='none', edgecolor=c,
                    linewidth=0.7, alpha=0.55, zorder=2
                ))

        # ── 2. Data points coloured by TRUE class ─────────────────────────
        for cls in [0, 1]:
            mask = y == cls
            ax.scatter(
                X_norm[mask, xi], X_norm[mask, yi],
                c=CLS_COLOR[cls], s=18, alpha=0.80,
                edgecolors='white', linewidths=0.3,
                zorder=4
            )

        # ── 3. Mark misclassifications (black X) ──────────────────────────
        y_pred_all = clf.predict(X)
        wrong = y_pred_all != y
        if wrong.any():
            ax.scatter(
                X_norm[wrong, xi], X_norm[wrong, yi],
                s=40, marker='x', c='#111111',
                linewidths=0.8, alpha=0.55, zorder=5,
                label='misclassified' if pidx == 0 else ''
            )

    # ── Cross-panel polylines (the "Shifted" part of SPC) ─────────────────
    # For every sample, connect its position in Panel k to Panel k+1.
    # Lines are drawn on the figure canvas using figure-fraction coordinates.
    for i in range(len(X_norm)):
        cls   = int(y[i])
        color = CLS_COLOR[cls]
        wrong_i = (clf.predict(X[[i]]) != y[[i]])[0]

        # Compute figure-space (x, y) for this sample in every panel
        fig_pts = []
        for pidx, (xi, yi) in enumerate(SPC_PANELS):
            xf, yf = data_to_fig(axes[pidx],
                                   X_norm[i, xi],
                                   X_norm[i, yi])
            fig_pts.append((xf, yf))

        # Draw connector between adjacent panels
        for k in range(n_panels - 1):
            x0f, y0f = fig_pts[k]
            x1f, y1f = fig_pts[k + 1]
            lw  = 0.8  if wrong_i else 0.35
            alp = 0.25 if wrong_i else 0.07
            lc  = '#000000' if wrong_i else color
            fig.add_artist(Line2D(
                [x0f, x1f], [y0f, y1f],
                transform=fig.transFigure,
                color=lc, alpha=alp,
                linewidth=lw, zorder=0,
                solid_capstyle='round'
            ))

    # ── Accuracy annotation ───────────────────────────────────────────────
    ann_parts = [f'max_depth={clf.max_depth}',
                  f'n_leaves={clf.get_n_leaves()}']
    if X_train is not None:
        tr_acc = accuracy_score(y_train, clf.predict(X_train))
        ann_parts.append(f'train_acc={tr_acc*100:.1f}%')
    if X_test is not None:
        te_acc = accuracy_score(y_test, clf.predict(X_test))
        ann_parts.append(f'test_acc={te_acc*100:.1f}%')

    # ── Legend ────────────────────────────────────────────────────────────
    legend_handles = [
        Line2D([0],[0], marker='o', color='w',
               markerfacecolor=CLS_COLOR[0], markersize=9,
               label='True: A wins (y=0)'),
        Line2D([0],[0], marker='o', color='w',
               markerfacecolor=CLS_COLOR[1], markersize=9,
               label='True: B wins (y=1)'),
        mpatches.Patch(facecolor=CLS_COLOR[0], alpha=0.42,
                       label='Tree region → A wins'),
        mpatches.Patch(facecolor=CLS_COLOR[1], alpha=0.42,
                       label='Tree region → B wins'),
        Line2D([0],[0], color='#555', linewidth=0.6, alpha=0.5,
               label='SPC connector (same sample)'),
        Line2D([0],[0], marker='x', color='#111', markersize=7,
               linewidth=0, markeredgewidth=0.9,
               label='Misclassified'),
    ]
    fig.legend(
        handles=legend_handles, loc='lower center', ncol=6,
        fontsize=8.5, framealpha=0.92, frameon=True,
        bbox_to_anchor=(0.5, -0.01), edgecolor='#ccc'
    )

    fig.suptitle(
        'SPC-DT — Kidney Recipient Pairwise Ranking\n' + '  |  '.join(ann_parts),
        fontsize=11, fontweight='bold', y=1.02, color='#222'
    )

    plt.savefig(save_path, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    print(f'Saved: {save_path}')
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: BONUS — SPC-DT DEPTH COMPARISON
# (same data, trees of depth 2 / 4 / 6 side by side in separate figures)
# ─────────────────────────────────────────────────────────────────────────────

def plot_depth_comparison(X_train, y_train, X_test, y_test, depths=(2, 4, 6)):
    """
    Train trees at different max_depth values, print accuracy,
    and save an SPC-DT visualisation for each depth.
    """
    print("\n" + "="*55)
    print("DEPTH COMPARISON")
    print("="*55)
    X_all = np.vstack([X_train, X_test])
    y_all = np.concatenate([y_train, y_test])

    for d in depths:
        clf = DecisionTreeClassifier(criterion='gini', max_depth=d,
                                      random_state=42)
        clf.fit(X_train, y_train)
        tr = accuracy_score(y_train, clf.predict(X_train))
        te = accuracy_score(y_test,  clf.predict(X_test))
        print(f'  depth={d}  leaves={clf.get_n_leaves():>3d}'
              f'  train={tr*100:.1f}%  test={te*100:.1f}%')
        plot_spc_dt(clf, X_all, y_all,
                    X_train=X_train, y_train=y_train,
                    X_test=X_test,   y_test=y_test,
                    save_path=f'spc_dt_depth{d}.png')


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT  (paste after your existing training code)
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == '__main__':

    # ── Reproduce your existing pipeline ────────────────────────────────
    data = np.loadtxt('data.csv', delimiter=',', skiprows=1)
    X    = data[:, :8]
    y    = data[:,  8].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = DecisionTreeClassifier(criterion='gini', max_depth=5,
                                    random_state=42)
    model.fit(X_train, y_train)
    y_pred   = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    error_pct = np.mean(y_pred != y_test) * 100

    print(f'Accuracy : {accuracy*100:.2f}%')
    print(f'Error %  : {error_pct:.2f}%')

    # ── SPC-DT for your trained model ───────────────────────────────────
    X_all = np.vstack([X_train, X_test])
    y_all = np.concatenate([y_train, y_test])

    plot_spc_dt(
        model, X_all, y_all,
        X_train=X_train, y_train=y_train,
        X_test=X_test,   y_test=y_test,
        save_path='spc_dt_visualization.png'
    )

    # ── Depth comparison (optional) ─────────────────────────────────────
    plot_depth_comparison(X_train, y_train, X_test, y_test, depths=[3, 5, 7])

Accuracy : 74.36%
Error %  : 25.64%
Saved: spc_dt_visualization.png

DEPTH COMPARISON
  depth=3  leaves=  8  train=78.2%  test=70.9%
Saved: spc_dt_depth3.png
  depth=5  leaves= 28  train=85.0%  test=74.4%
Saved: spc_dt_depth5.png
  depth=7  leaves= 66  train=90.4%  test=77.8%
Saved: spc_dt_depth7.png
